In [6]:
1e8

100000000.0

In [5]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 1e6 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [02:44,  9.13it/s, step size=1.89e-02, acc. prob=0.924]


In [7]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 1e-3 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [00:29, 50.44it/s, step size=4.44e-02, acc. prob=0.919]


In [8]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 1e-2 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [01:33, 16.13it/s, step size=3.86e-02, acc. prob=0.939]


In [9]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 1e-1 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [02:15, 11.07it/s, step size=3.34e-02, acc. prob=0.936]


In [10]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 1 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [02:58,  8.40it/s, step size=1.66e-02, acc. prob=0.948]


In [11]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import plotly.graph_objects as go
import numpy as np

# Generate synthetic data using a combination of sine and cosine functions
np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)  # X values

# Combining sine and cosine functions to create a synthetic signal
y_true = np.sin(X) + 0.5 * np.cos(2 * X) + 0.3 * np.sin(3 * X)
y = y_true + np.random.normal(0, 0.1, size=n)  # Adding some noise

# Convert data to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float).unsqueeze(1)  # Shape (n, 1)
y_tensor = torch.tensor(y, dtype=torch.float)

# Define the basis functions {1, x, x^2, x^3}
def basis_functions(X):
    return torch.stack([torch.ones_like(X), X, X**2, X**3], dim=1)

# Define Bayesian Linear Basis Regression Model using Pyro
def bayesian_linear_basis_regression(X, y=None):
    # Priors for weights and intercept for each basis function
    w = pyro.sample("w", dist.Normal(torch.zeros(4), 10 * torch.ones(4)))  # Weights for {1, x, x^2, x^3}
    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))  # Prior for noise
    
    # Linear combination of basis functions
    Phi_X = basis_functions(X)  # Design matrix
    mean = torch.matmul(Phi_X, w)  # Linear model: w0 + w1*x + w2*x^2 + w3*x^3
    
    # Likelihood
    with pyro.plate("data", X.shape[0]):
        pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

# NUTS sampler
nuts_kernel = NUTS(bayesian_linear_basis_regression)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=500)
mcmc.run(X_tensor[:, 0], y_tensor)

# Extract samples from the posterior
posterior_samples = mcmc.get_samples()

# Convert to numpy arrays for plotting
w_samples = posterior_samples['w'].numpy()

# Generate the predicted regression lines from posterior samples
X_plot = np.linspace(0, 10, 100)  # For plotting smooth curve
X_plot_tensor = torch.tensor(X_plot, dtype=torch.float).unsqueeze(1)
Phi_X_plot = basis_functions(X_plot_tensor).numpy().reshape(100, 4)  # Fix the shape

# Generate regression lines using posterior samples
y_pred_samples = np.dot(Phi_X_plot, w_samples.T)  # Matrix multiplication

# Plot the original data and the posterior predictive regression lines
fig = go.Figure()

# Scatter plot of the original data
fig.add_trace(go.Scatter(x=X, y=y, mode='markers', name='Data'))

# Plot regression lines for a subset of posterior samples
for i in range(100):  # Plot 100 regression lines from posterior samples
    fig.add_trace(go.Scatter(x=X_plot, y=y_pred_samples[:, i], mode='lines', line=dict(width=0.5, color='blue'), opacity=0.2, showlegend=False))

# Update layout
fig.update_layout(
    title="Bayesian Linear Basis Regression (Training on Sine/Cosine Data)",
    xaxis_title="X",
    yaxis_title="y",
)

# Show plot
fig.show()

Sample: 100%|██████████| 1500/1500 [02:47,  8.95it/s, step size=2.20e-02, acc. prob=0.924]
